# Save surface clusters with Python

This notebook extracts connected suprathreshold clusters from a cortical surface map. It applies two thresholds:

1. Vertex-level threshold: `z > Z_THRESHOLD`.
2. Cluster-level threshold: at least `MIN_CLUSTER_VERTICES` connected vertices.

It saves the full suprathreshold mask, a z-value map and binary mask for each retained cluster, the corresponding masked original-statistic maps, and their combined map.

Xiao Chen, 2026-06-10  
chenxiaophd@gmail.com


In [ ]:
from pathlib import Path

import nibabel as nib
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from nilearn import datasets, plotting

%load_ext autoreload
%autoreload 2
import surface_utils as su


## Configuration


In [ ]:
# where the lesion maps stored
RESULT_DIR = Path("~/rumination_lesion_network_mapping/maps")
TEMPLATE_DIR = Path("/dpabi_dir/DPABISurf/SurfTemplates")
OUTPUT_DIR = Path("xxxxxx")

Z_THRESHOLD = 3.29
MIN_CLUSTER_VERTICES = 20
HEMISPHERE = "rh"

# cureent results: GSR
Z_MAP_PATH = RESULT_DIR / f"z_rumination_MDD_GSR_{HEMISPHERE}.gii"
STAT_MAP_PATH = RESULT_DIR / f"rumination_MDD_GSR_{HEMISPHERE}.gii"
SURFACE_PATH = TEMPLATE_DIR / f"fsaverage5_{HEMISPHERE}_inflated.surf.gii"


## Load the surface data


In [ ]:
z_gii = nib.load(Z_MAP_PATH)
z_data = np.asarray(z_gii.darrays[0].data, dtype=float)
n_vertices = z_data.size

surface_gii = nib.load(SURFACE_PATH)
faces = np.asarray(surface_gii.darrays[1].data, dtype=np.int64)

if faces.max() >= n_vertices:
    raise ValueError("The surface mesh contains vertex indices outside the z-map range.")


## Build the vertex adjacency matrix


In [ ]:
def build_surface_adjacency(surface_faces, n_vertices):
    """Build a symmetric sparse adjacency matrix from triangular faces."""
    edges = np.vstack(
        [
            surface_faces[:, [0, 1]],
            surface_faces[:, [1, 2]],
            surface_faces[:, [2, 0]],
        ]
    )
    row, col = edges.T
    adjacency = coo_matrix(
        (np.ones(row.size, dtype=np.uint8), (row, col)),
        shape=(n_vertices, n_vertices),
    ).tocsr()
    return adjacency.maximum(adjacency.T)


adjacency = build_surface_adjacency(faces, n_vertices)


In [ ]:
def save_gifti_array(values, output_path):
    """Save a one-dimensional float array as a GIFTI functional image."""
    values = np.asarray(values, dtype=np.float32)
    image = nib.gifti.GiftiImage(
        darrays=[
            nib.gifti.GiftiDataArray(
                values,
                datatype="NIFTI_TYPE_FLOAT32",
            )
        ]
    )
    nib.save(image, output_path)


def save_clusters(
    mask,
    values,
    adjacency,
    prefix,
    min_vertices=20,
    output_dir=Path("."),
    save_total_mask=True,
):
    """Save connected clusters that meet the minimum vertex-count threshold."""
    mask = np.asarray(mask, dtype=bool)
    values = np.asarray(values)
    output_dir = Path(output_dir)

    if mask.ndim != 1 or values.ndim != 1 or mask.size != values.size:
        raise ValueError("mask and values must be one-dimensional arrays of equal length.")
    if adjacency.shape != (mask.size, mask.size):
        raise ValueError("The adjacency matrix shape does not match the surface data.")
    if min_vertices < 1:
        raise ValueError("min_vertices must be at least 1.")

    output_dir.mkdir(parents=True, exist_ok=True)

    if save_total_mask:
        total_mask_path = output_dir / f"{prefix}_suprathreshold_mask.func.gii"
        save_gifti_array(mask, total_mask_path)

    suprathreshold_indices = np.flatnonzero(mask)
    if suprathreshold_indices.size == 0:
        print(f"[{prefix}] No suprathreshold vertices.")
        return []

    subgraph = adjacency[suprathreshold_indices][:, suprathreshold_indices]
    n_components, component_labels = connected_components(
        csgraph=subgraph,
        directed=False,
    )

    # Sort retained clusters by size so output numbering is deterministic.
    components = []
    for component_id in range(n_components):
        component_vertices = suprathreshold_indices[component_labels == component_id]
        if component_vertices.size >= min_vertices:
            components.append(component_vertices)
    components.sort(key=len, reverse=True)

    cluster_outputs = []
    for cluster_number, cluster_vertices in enumerate(components, start=1):
        cluster_size = cluster_vertices.size
        cluster_mask = np.zeros(mask.size, dtype=bool)
        cluster_mask[cluster_vertices] = True

        cluster_z_values = np.zeros(values.size, dtype=np.float32)
        cluster_z_values[cluster_mask] = values[cluster_mask]

        stem = f"{prefix}_cluster{cluster_number}_n{cluster_size}"
        z_path = output_dir / f"{stem}_z.func.gii"
        mask_path = output_dir / f"{stem}_mask.func.gii"
        save_gifti_array(cluster_z_values, z_path)
        save_gifti_array(cluster_mask, mask_path)

        cluster_outputs.append(
            {
                "cluster": cluster_number,
                "n_vertices": cluster_size,
                "z_path": z_path,
                "mask_path": mask_path,
            }
        )

    print(
        f"[{prefix}] Saved {len(cluster_outputs)} clusters "
        f"with at least {min_vertices} vertices."
    )
    return cluster_outputs


## Extract and save significant clusters


In [ ]:
positive_mask = np.isfinite(z_data) & (z_data > Z_THRESHOLD)
threshold_label = f"{Z_THRESHOLD:g}".replace(".", "p")
output_prefix = f"positive_gt{threshold_label}"

cluster_outputs = save_clusters(
    mask=positive_mask,
    values=z_data,
    adjacency=adjacency,
    prefix=output_prefix,
    min_vertices=MIN_CLUSTER_VERTICES,
    output_dir=OUTPUT_DIR,
)


## Apply each cluster mask to the original statistic map


In [ ]:
for cluster_output in cluster_outputs:
    stats_path = cluster_output["mask_path"].with_name(
        cluster_output["mask_path"].name.replace("_mask.func.gii", "_stats.func.gii")
    )
    su.apply_surface_mask(
        stat_gii=STAT_MAP_PATH,
        mask_gii=cluster_output["mask_path"],
        out_gii=stats_path,
    )
    cluster_output["stats_path"] = stats_path


## Combine the retained cluster maps


In [ ]:
stat_gii = nib.load(STAT_MAP_PATH)
stat_data = np.asarray(stat_gii.darrays[0].data, dtype=np.float32)

if stat_data.size != n_vertices:
    raise ValueError("The original statistic map and z-map have different vertex counts.")

combined_mask = np.zeros(n_vertices, dtype=bool)
for cluster_output in cluster_outputs:
    cluster_mask_gii = nib.load(cluster_output["mask_path"])
    combined_mask |= np.asarray(cluster_mask_gii.darrays[0].data, dtype=bool)

combined_stats = np.zeros(n_vertices, dtype=np.float32)
combined_stats[combined_mask] = stat_data[combined_mask]
combined_path = OUTPUT_DIR / f"{output_prefix}_sig_clusters.func.gii"
save_gifti_array(combined_stats, combined_path)

print(f"Saved combined cluster map to: {combined_path}")


## Optional: interactively inspect the thresholded z-map


In [ ]:
z_data_clean = np.nan_to_num(z_data, nan=0.0)
fsaverage = datasets.fetch_surf_fsaverage(mesh="fsaverage5")

hemisphere_surface = {
    "lh": fsaverage.infl_left,
    "rh": fsaverage.infl_right,
}[HEMISPHERE]

view = plotting.view_surf(
    surf_mesh=hemisphere_surface,
    surf_map=z_data_clean,
    threshold=Z_THRESHOLD,
    cmap="cold_hot",
    colorbar=True,
)

view
